# First null-subj-v2 eval wave — 2026-06-11

First all-cells eval pass (EN, slot h0-s42, scoring_version `null-subj-v2-r1`,
97 checkpoints/run). Run while the training fleet was still finishing the
h2-h4 layers.

**Caveats for this wave (r1):**
- `ext_subj` items (48) were scored with a literal `"nan"` pseudo-context
  (empty CSV context cell through `str()`). Fixed in `212ef16`+; rescore as
  `null-subj-v2-r2` after the post-wave stimulus adjustments. Pair contrasts
  remain internally consistent but treat `extraction` as suspect.
- One run per cell (h0-s42) — no seed error bars yet.
- `control` / same-subject `conjunction` categories are *designed* so the
  null variant is grammatical: low overt-preference there is success.

Data refresh: `AWS_PROFILE=nrp python scripts/pull_eval_results.py`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path(__file__).resolve().parents[3] if "__file__" in dir() else Path.cwd().resolve().parents[2]
DATA = REPO / "data/eval_results/null_subj_v2"
FIGS = REPO / "analysis/eval_v2/figures"
FIGS.mkdir(parents=True, exist_ok=True)

pairs = pd.concat([pd.read_parquet(f) for f in (DATA / "pairs").glob("*.parquet")])
ckpts = pd.concat([pd.read_parquet(f) for f in (DATA / "checkpoints").glob("*.parquet")])
print(f"cells: {pairs.cell_id.nunique()}  pair-rows: {len(pairs):,}")
pairs.cell_id.unique()

## Epoch boundaries (from the checkpoints sidecar)
First checkpoint step inside each epoch; used as vertical annotations below.

In [ ]:
epoch_bounds = ckpts.groupby("epoch").checkpoint_step.min()
EP1, EP2 = int(epoch_bounds.get(1, 1044)), int(epoch_bounds.get(2, 2032))
epoch_bounds.head(5)

## Ablation-contrast trajectories
Headline observations from the first wave (gpt2_small):
- the grammatical phase transition completes **within epoch 1**; epochs 2-30 are flat
- `lemmatize_verbs` selectively degrades 3rd-person subject conditions + `expl_seems`
- `impoverish_case` zeroes `obj_3pl` ("them" detrained) — the manipulation, visible at item level
- `remove_expletive_sentences` still reaches ceiling on `expletive` (indirect evidence?)

In [ ]:
conds = ["baseline", "remove_expletive_sentences", "impoverish_case", "lemmatize_verbs",
         "enrich_verbal_morphology"]
labels = {"baseline": "baseline", "remove_expletive_sentences": "\u2212 expletives",
          "impoverish_case": "\u2212 case", "lemmatize_verbs": "\u2212 verb morph",
          "enrich_verbal_morphology": "+ verb morph"}
colors = dict(zip(conds, ["black", "#d62728", "#1f77b4", "#2ca02c", "#ff7f0e"]))


def ablation_figure(pairs, arch, cats, fname=None):
    sub_arch = pairs[pairs.architecture == arch]
    fig, axes = plt.subplots(1, len(cats), figsize=(4 * len(cats), 3.8), sharey=True)
    for ax, cat in zip(axes, cats):
        for cond in conds:
            sub = sub_arch[(sub_arch.intervention == cond) & (sub_arch.category == cat)]
            if sub.empty:
                continue
            traj = sub.groupby("checkpoint_step").prefers_overt_meanlp.mean()
            ax.plot(traj.index + 1, traj.values, label=labels[cond], color=colors[cond], lw=1.7)
        for x, lbl in [(EP1, "epoch 1"), (EP2, "epoch 2")]:
            ax.axvline(x, color="purple", ls="--", lw=1, alpha=0.6)
            ax.text(x, 0.04, f" {lbl}", color="purple", fontsize=7.5, rotation=90, va="bottom")
        ax.set_xscale("log")
        ax.axhline(0.5, color="gray", ls=":", lw=1)
        ax.set_title(cat, fontsize=10)
        ax.set_xlabel("optimizer step + 1 (log)")
        ax.set_ylim(0, 1.05)
    axes[0].set_ylabel("P(prefers overt) [MeanLP]")
    axes[0].legend(fontsize=8, loc="center left")
    fig.suptitle(f"{arch} en \u2014 ablation contrasts (h0-s42)", y=1.04)
    fig.tight_layout()
    if fname:
        fig.savefig(FIGS / fname, dpi=150, bbox_inches="tight")
    return fig


ablation_figure(pairs, "gpt2_small",
                ["subject_drop", "subject_drop_no_agreement", "expletive", "object_drop"],
                fname="gpt2small_ablation_epochs.png");

In [ ]:
# Replication at the next size up: enrich_verbal_morphology depressed/unstable,
# impoverish_case flattens object_drop (obj_3pl), transition still inside epoch 1.
ablation_figure(pairs, "gpt2_medium",
                ["subject_drop", "subject_drop_no_agreement", "expletive", "object_drop"],
                fname="gpt2medium_ablation_epochs.png");

## By-item breakdown (final checkpoint per cell)
Final step differs per condition (line-removal ablations have fewer chunks/epoch),
so "final" is per-cell max.

In [ ]:
finals = pairs.groupby("cell_id").checkpoint_step.transform("max")
final = pairs[pairs.checkpoint_step == finals].copy()
tab = final.pivot_table(index=["architecture", "category", "condition"],
                        columns="intervention",
                        values="prefers_overt_meanlp", aggfunc="mean").round(2)
tab

In [ ]:
# Uniformity (Jaeggli & Safir / Hyams) vs recoverability: both morphology
# ablations lower overt-preference (uniformly-rich AND uniformly-bare both
# license null subjects under uniformity), while non-morphological ablations
# (−expletives, −case) leave subject_drop untouched — arguing against a
# generic perturbation account. The person profile then dissociates the two:
# lemma's effect concentrates in 3rd person (where English's only agreement
# cue, -s, lived) — a local cue-loss signature; enrich's effect is
# person-broad — a global licensing change. Tail drift shows both effects
# persist/strengthen at 30 epochs (not convergence lag). n=1 seed: replicate
# on h1-h4 × s137 before believing any of this.
sd = pairs[pairs.category.isin(["subject_drop", "subject_drop_no_agreement"])]
gm = sd[sd.architecture.isin(["gpt2_small", "gpt2_medium"])]
fin_steps = gm.groupby("cell_id").checkpoint_step.transform("max")
person = (gm[gm.checkpoint_step == fin_steps]
          .pivot_table(index=["architecture", "condition"], columns="intervention",
                       values="prefers_overt_meanlp", aggfunc="mean")
          [["baseline", "lemmatize_verbs", "enrich_verbal_morphology"]].round(2))
display(person)
for arch in ["gpt2_small", "gpt2_medium"]:
    for cond in ["baseline", "lemmatize_verbs", "enrich_verbal_morphology"]:
        traj = (gm[(gm.architecture == arch) & (gm.intervention == cond)]
                .groupby("checkpoint_step").prefers_overt_meanlp.mean())
        early = traj[(traj.index >= 5000) & (traj.index <= 15000)].mean()
        late = traj[traj.index >= 25000].mean()
        print(f"{arch:<12} {cond:<26} ep5-15:{early:.3f} ep25-30:{late:.3f} drift:{late-early:+.3f}")

In [ ]:
# Items unanimously preferring null across models of one arch — join to stimulus text.
# (For control/conjunction that's correct behavior; elsewhere it's worth reading.)
arch = "gpt2_small"
fa = final[final.architecture == arch]
unanim = (fa.groupby(["category", "condition", "item_id"]).prefers_overt_meanlp.mean())
worst = unanim[unanim == 0.0]
rows = []
for (cat, cond, iid) in worst.index:
    csv = REPO / f"evaluation/stimuli/null-subj-v2/staging/en/{cat}.csv"
    if csv.exists():
        s = pd.read_csv(csv)
        m = s[(s.item_id == iid) & (s.condition == cond) & (s.pronoun_status == 1)]
        for _, r in m.iterrows():
            ctx = "" if pd.isna(r.context) else r.context
            rows.append((cat, cond, iid, f"{ctx} | {r.target}"))
pd.DataFrame(rows, columns=["category", "condition", "item_id", "overt variant"])